# ECG Torch Renderer 0.4.0: GPU benchmark and profiling

Choose **Runtime > Change runtime type > GPU**, then use **Run all**.
This notebook uses artificial waveforms, without mounting Drive or loading patient data.
The final ZIP includes tests, benchmarks, profiling reports and generated examples.
The first compiled call can take several minutes and is measured separately.
Small CNN and U-Net checks verify software integration, not clinical accuracy.


In [ ]:
import json, os, subprocess, sys, time, zipfile
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Runtime > Change runtime type, then run again.")
GPU = torch.cuda.get_device_properties(0)
print({"torch": str(torch.__version__), "gpu": GPU.name,
       "vram_GiB": GPU.total_memory / 2**30,
       "compute_capability": torch.cuda.get_device_capability(0)})

REPOSITORY = "fmenegui/ecg-torch-renderer"
BRANCH = "main"
CHECKOUT = Path.cwd() / "ecg-renderer-source"
RESULTS = Path.cwd() / ("ecg-colab-results-" + time.strftime("%Y%m%d-%H%M%S"))
RESULTS.mkdir(exist_ok=False)

if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "--branch", BRANCH, "https://github.com/" + REPOSITORY + ".git",
                    str(CHECKOUT)], check=True)
PACKAGE = CHECKOUT
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CHECKOUT, text=True).strip()
# Preserve the CUDA Torch build supplied by Colab.
subprocess.run([sys.executable, "-m", "pip", "install", "numpy>=1.24", "Pillow>=10.1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", str(PACKAGE)], check=True)
import ecg_gpu
print({"version": ecg_gpu.__version__, "commit": commit})
(RESULTS / "runtime.json").write_text(json.dumps({
    "repository": REPOSITORY, "branch": BRANCH, "commit": commit,
    "version": ecg_gpu.__version__, "torch": str(torch.__version__),
    "cuda_build": torch.version.cuda, "gpu": GPU.name,
    "vram_bytes": GPU.total_memory, "artificial_signals": True}, indent=2))

In [ ]:
# Edit these controls as needed. Start with batch sizes 1 and 4.
BATCHES = [1, 4]
PRESETS = ["fast", "balanced", "full"]
STEPS = 30
WARMUP = 8
PROFILE_STEPS = 3
EXPORT_MEMORY_TIMELINE = False  # True includes stacks and increases profiling overhead.
FAILURES = []

def run_report(name, arguments, cwd=None):
    log = RESULTS / (name + ".log")
    print("Running", name, flush=True)
    with log.open("w") as stream:
        process = subprocess.run([sys.executable, *arguments], cwd=cwd,
                                 stdout=stream, stderr=subprocess.STDOUT)
    lines = log.read_text(errors="replace").splitlines()
    print("\n".join(lines[-12:]))
    if process.returncode:
        FAILURES.append({"test": name, "returncode": process.returncode})
    return process.returncode == 0

run_report("software-tests", ["-m", "unittest", "discover", "-s", "tests", "-v"], cwd=PACKAGE)

In [ ]:
# Every backend receives the same signals and seed sequence.
# Backend order alternates. Compilation is excluded from warm timings.
run_report("paired-comparison", ["-m", "ecg_gpu.compare",
    "--legacy", str(PACKAGE / "references" / "ecg-gpu-generator-v021.zip"),
    "--device", "cuda", "--out", str(RESULTS / "paired-comparison"),
    "--presets", *PRESETS, "--batches", *map(str, BATCHES),
    "--steps", str(STEPS), "--warmup", str(WARMUP)])
path = RESULTS / "paired-comparison" / "comparison.json"
if path.exists():
    report = json.loads(path.read_text())
    for case in report["cases"]:
        print({"preset": case["preset"], "B": case["batch_size"],
               "images_s": {k: round(v["images_s"], 2) for k, v in case["backends"].items()},
               "speedup": round(case["compiled_speedup_vs_v021"], 2),
               "pixel_error": case["max_rgb_error_vs_v021"]})

In [ ]:
# Separate processes keep backend memory and compilation independent.
for backend, flags in [("eager", []), ("compiled", ["--compile-raster"])]:
    run_report("profile-" + backend, ["-m", "ecg_gpu.profile",
        "--device", "cuda", "--preset", "full", "--batch-size", "4",
        "--steps", str(STEPS), "--warmup", str(WARMUP),
        "--profile-steps", str(PROFILE_STEPS),
        "--out", str(RESULTS / ("profile-" + backend)), *flags,
        *(["--memory-timeline"] if EXPORT_MEMORY_TIMELINE else [])])
run_report("on-the-fly-training-profile", ["-m", "ecg_gpu.profile",
    "--device", "cuda", "--preset", "balanced", "--batch-size", "4",
    "--steps", str(STEPS), "--warmup", str(WARMUP),
    "--profile-steps", str(PROFILE_STEPS), "--compile-raster", "--train-step",
    "--out", str(RESULTS / "on-the-fly-training")])

In [ ]:
# Exercise aligned 12-channel supervision and the U-Net training example.
run_report("unet-supervision", ["examples/segmentation.py", "--device", "cuda",
    "--compile-raster", "--steps", "20"], cwd=PACKAGE)


In [ ]:
from PIL import Image
from ecg_gpu import ECGImageLayer, RenderConfig
from ecg_gpu.demo import demo_signals
from IPython.display import display

values, _, _ = demo_signals(fs=500, duration=10)
signals = torch.tensor(values, dtype=torch.float32, device="cuda")[None]
layer = ECGImageLayer(fs=500, config=RenderConfig(height=768, width=1024), seed=1973).cuda()
common_capture = {"quarter": 0, "zoom": .82, "angle": .04,
    "translate_x": 0., "translate_y": 0., "surface_strength": .95,
    "relief_view": [.52, -.30], "paper_palette": 0, "paper_tint": 0.,
    "paper_color": [1., 1., 1.], "paper_age": 0., "grid_strength": .30,
    "minor_strength": .27, "line_width": 1.25, "ink_color": [.065, .060, .058],
    "ink_density": .96, "lead_present": 1., "pulse_present": 1.,
    "header_present": 1., "object_shadow": "none", "shadow_strength": 0.,
    "coffee_strength": 0., "stain_strength": 0., "band_strength": 0.,
    "artifact_present": 0., "occlusion_active": 0., "pen_active": 0.}
captures = [
    {"layout": "3x4+r3", "surface_mode": "crumpled", "background_type": "wood"},
    {"layout": "6x2", "surface_mode": "fold", "background_type": "marble"},
    {"layout": "12x1", "surface_mode": "curl", "background_type": "fabric"},
]
for index, parameters in enumerate(captures):
    image = layer(signals, parameters={**common_capture, **parameters})[0].permute(1, 2, 0).cpu().numpy()
    preview = Image.fromarray((image.clip(0, 1) * 255).round().astype("uint8"))
    preview.save(RESULTS / ("example-" + str(index) + ".png"))
    display(preview.resize((768, 576)))
del signals, layer
torch.cuda.empty_cache()

In [ ]:
from ecg_gpu import ECGSupervisionLayer, SupervisionConfig, profile_layer

values, _, _ = demo_signals(fs=500, duration=10)
signals = torch.tensor(values, dtype=torch.float32, device="cuda")[None].expand(4, -1, -1)
synth = ECGSupervisionLayer(fs=500, config=RenderConfig(height=384, width=512),
    compile_raster=True, supervision=SupervisionConfig(outputs=("lead_masks", "lead_boxes"))).cuda()
try:
    supervision_report = profile_layer(synth, signals, RESULTS / "profile-supervision",
        steps=STEPS, warmup=WARMUP, profile_steps=PROFILE_STEPS,
        workload_name="ECGSupervisionLayer.forward")
    print(supervision_report["timing"])
except Exception as error:
    FAILURES.append({"test": "profile-supervision", "error": str(error)})
finally:
    del synth, signals
    torch.cuda.empty_cache()


In [ ]:
# Preserve logs and partial results even if one benchmark fails.
(RESULTS / "failures.json").write_text(json.dumps(FAILURES, indent=2))
archive = RESULTS.with_suffix(".zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as package:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file() and path.suffix != ".zip":
            package.write(path, str(path.relative_to(RESULTS)))
print({"zip": str(archive), "failures": FAILURES})
from google.colab import files
files.download(str(archive))

Send the results ZIP for bottleneck analysis. `report.json` records warm latency,
p90, allocator memory, shapes, regions and kernels. Decompress `trace.json.gz`
and open it in https://ui.perfetto.dev.
If `cuda_kernel_activity_captured` is false, CUDA activity capture was unavailable
in that runtime. The report records this separately from synchronized timing and
CUDA Events. A failed test remains a failure, even when partial results are exported.

References: https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html
and https://research.google.com/colaboratory/faq.html.
